# Analysis Notebook 

1. evaluates faithfulness (insertion/deletion AUC)
2. provides optional ablation comparison hooks
3. runs mandatory Grad-CAM++ diagnostics

In [ ]:
from __future__ import annotations

from itertools import combinations
from pathlib import Path
import json
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if not SRC_ROOT.exists():
    raise FileNotFoundError(f"Could not find src/ under {PROJECT_ROOT}")
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from cv.analysis import (
    classify_gradcampp_outcome,
    compute_method_deltas,
    compute_primary_correct_intersection,
    deletion_auc,
    insertion_auc,
    paired_bootstrap_ci,
    paired_permutation_pvalue,
    run_insertion_deletion,
    summarize_condition_level_deltas,
    summarize_condition_level_metrics,
    summarize_seed_level_deltas,
    summarize_seed_level_metrics,
)
from cv.config import ARTIFACTS_ROOT, DEFAULT_ENCODER_CHECKPOINTS
from cv.data import load_eval_subset, load_stl10_split
from cv.explain import discover_stage4_runs, load_saliency_map, read_saliency_metadata
from cv.models import build_downstream_model, resolve_mode_config
from cv.train.trainer import TrainingRunConfig, train_one_run
from cv.transforms import build_eval_transform
from cv.utils.io import ensure_parent, read_json, write_json


In [ ]:
RUN_CONFIG: dict[str, object] = {
    "artifacts_root": None, # path for saving outputs, defaults to artifacts
    "data_root": None, # optional override for STL-10 data 
    "device": "cpu", # device used for model loading and inference for perturbation evals   
    "allow_remote_download": False, # do not download encoders again for eval
    "conditions": ["supervised", "moco", "swav", "random_init"],
    "seeds": [0, 1, 2],
    "methods": ["gradcam", "gradcampp", "occlusion"], # saliency methods to evaluate 
    "patch_size": 16,
    "stride": 16,
    "blur_kernel_size": 21, # gaussian blur kernel size for making blurred baseline 
    # and perturbations, larger = stronger smoothing 
    "blur_sigma": 5.0, # std for gaussian blur 
    "eval_batch_size": 64, # model batch size 
    "bootstrap_resamples": 1000, # number of paired bootstrap resamples for confidence intervals  
    "permutation_resamples": 5000, # number of resamples for paired permutation p values for insertion 
    # and deletion AUC
    "gradcampp_effect_threshold": 0.01, # for deciding meaningfulness of gradCAM++ values 
    
    # ablation configs 
    "run_stage8_ablation": True, # ablation fine tuning flag 
    "stage8_conditions": ["supervised", "moco", "swav"],
    "stage8_seed": 0, 
    "stage8_recipe_id": "ablation_layer4_v1", # hyperparam config for ablation 
    # (layer4 + classifier training). NOTE: encoders are not retrained and remain frozen 
    "stage8_num_workers": 0, # dataloader worker config 
    "stage8_pin_memory": False, # false for CPU
    "stage8_sanity_checks": True, # gradient/BN/training check before full fine tuning 
    "primary_slice_policy": "all_conditions_correct_intersection", # for permutation slices 
    "stage9_panel_seed": 0,
    "stage9_panel_reference_condition": "supervised", # reference for picking IDs 
    "stage9_panel_num_images": 3, # number of image IDs to include for plotting 
    "stage9_panel_image_ids": None, # auto select
}

ARTIFACTS_ROOT_RESOLVED = Path(RUN_CONFIG["artifacts_root"]) if RUN_CONFIG["artifacts_root"] else ARTIFACTS_ROOT
print(json.dumps(RUN_CONFIG, indent=2))
print(f"Resolved artifacts root: {ARTIFACTS_ROOT_RESOLVED}")


In [ ]:
def _load_stage4_model(*, run, device: str, allow_remote_download: bool):
    mode_config = resolve_mode_config(condition=run.condition, training_mode=run.training_mode)

    checkpoint_path = None
    if run.condition == "moco":
        checkpoint_path = str(DEFAULT_ENCODER_CHECKPOINTS.moco_checkpoint_path)
    elif run.condition == "swav":
        checkpoint_path = str(DEFAULT_ENCODER_CHECKPOINTS.swav_checkpoint_path)

    model = build_downstream_model(
        condition=run.condition,
        num_classes=10,
        freeze_encoder=mode_config.freeze_encoder,
        trainable_layer4=mode_config.trainable_layer4,
        device=torch.device(device),
        allow_remote_download=allow_remote_download,
        checkpoint_path=checkpoint_path,
    )

    payload = torch.load(run.checkpoint_path, map_location=torch.device(device), weights_only=False)
    model.load_state_dict(payload["model_state_dict"])
    model.eval()
    return model


def _metadata_path(*, artifacts_root: Path, condition: str, seed: int, method: str) -> Path:
    return artifacts_root / "saliency" / condition / f"seed_{seed}" / method / "metadata.json"


def _resolve_map_path(path_value: str, artifacts_root: Path) -> Path:
    path = Path(path_value)
    if path.is_absolute():
        return path
    return artifacts_root / path


def _index_rows_by_image_id(rows: list[dict[str, object]]) -> dict[int, dict[str, object]]:
    indexed: dict[int, dict[str, object]] = {}
    for row in rows:
        image_id = row.get("test_image_id")
        if not isinstance(image_id, int):
            raise ValueError(f"Row missing integer test_image_id: {row}")
        if image_id in indexed:
            raise ValueError(f"Duplicate test_image_id in metadata: {image_id}")
        indexed[image_id] = row
    return indexed


## Stage 7: Make Subsets for AUC summaries and permutation tests 

This notebook uses 2 different slices for evals, `all200`, all images in the fixed eval subset, and `primary`, a stricter subset where for a given seed, the image was correctly classified by all conditions that are being compared against each other (all models identified this sample correctly).

- loads fixed eval subset IDs
- load saliency metadata for each condition, seed and method
- find the correctly identified samples per seed ("for each seed X models, find the samples where all models correctly classified it, and get the ID for the sample to create the `primary` subset").
- stores the IDs as `primary_ids_by_seed`

In [ ]:
subset_artifacts = load_eval_subset(artifacts_root=RUN_CONFIG["artifacts_root"])
subset_ids = sorted(int(idx) for idx in subset_artifacts.indices)

runs = discover_stage4_runs(
    artifacts_root=RUN_CONFIG["artifacts_root"],
    conditions=list(RUN_CONFIG["conditions"]),
    seeds=[int(seed) for seed in list(RUN_CONFIG["seeds"])],
)
run_lookup = {(run.condition, int(run.seed)): run for run in runs}

metadata_rows: list[dict[str, object]] = []
metadata_by_key: dict[tuple[str, int, str], list[dict[str, object]]] = {}
for condition in list(RUN_CONFIG["conditions"]):
    for seed in [int(s) for s in list(RUN_CONFIG["seeds"])]:
        if (condition, seed) not in run_lookup:
            raise FileNotFoundError(f"Missing Stage-4 run metrics for {condition}/seed_{seed}.")
        for method in list(RUN_CONFIG["methods"]):
            metadata_path = _metadata_path(
                artifacts_root=ARTIFACTS_ROOT_RESOLVED,
                condition=condition,
                seed=seed,
                method=method,
            )
            if not metadata_path.exists():
                raise FileNotFoundError(f"Missing saliency metadata: {metadata_path}")
            rows = read_saliency_metadata(metadata_path)
            metadata_by_key[(condition, seed, method)] = rows
            metadata_rows.extend(rows)

primary_ids_by_seed = compute_primary_correct_intersection(
    metadata_rows,
    conditions=list(RUN_CONFIG["conditions"]),
    seeds=[int(seed) for seed in list(RUN_CONFIG["seeds"])],
)

primary_counts_df = pd.DataFrame(
    [
        {"seed": seed, "n_primary": len(ids), "n_all": len(subset_ids)}
        for seed, ids in sorted(primary_ids_by_seed.items())
    ]
)
display(primary_counts_df)
print(f"Fixed eval subset size: {len(subset_ids)}")


## Stage 7 - Per-Image AUC Computation and Aggregation


In [ ]:
test_dataset = load_stl10_split(
    "test",
    transform=build_eval_transform(),
    data_root=RUN_CONFIG["data_root"],
    download=False,
)

per_image_rows: list[dict[str, object]] = []
for condition in list(RUN_CONFIG["conditions"]):
    for seed in [int(s) for s in list(RUN_CONFIG["seeds"])]:
        print(f"[eval] condition={condition} seed={seed}")
        run = run_lookup[(condition, seed)]
        model = _load_stage4_model(
            run=run,
            device=str(RUN_CONFIG["device"]),
            allow_remote_download=bool(RUN_CONFIG["allow_remote_download"]),
        )

        primary_ids = set(primary_ids_by_seed[seed])
        for method in list(RUN_CONFIG["methods"]):
            method_rows = metadata_by_key[(condition, seed, method)]
            rows_by_image = _index_rows_by_image_id(method_rows)

            missing_ids = sorted(set(subset_ids) - set(rows_by_image))
            if missing_ids:
                raise ValueError(
                    f"Missing metadata rows for {condition}/seed_{seed}/{method}: {missing_ids[:5]}..."
                )

            for image_id in subset_ids:
                row = rows_by_image[image_id]
                image, _ = test_dataset[image_id]
                map_path = _resolve_map_path(str(row["saliency_map_path"]), ARTIFACTS_ROOT_RESOLVED)
                saliency = load_saliency_map(map_path)

                result = run_insertion_deletion(
                    model=model,
                    image=image,
                    saliency=saliency,
                    target_class=int(row["predicted_class"]),
                    patch_size=int(RUN_CONFIG["patch_size"]),
                    stride=int(RUN_CONFIG["stride"]),
                    blur_kernel_size=int(RUN_CONFIG["blur_kernel_size"]),
                    blur_sigma=float(RUN_CONFIG["blur_sigma"]),
                    eval_batch_size=int(RUN_CONFIG["eval_batch_size"]),
                    device=str(RUN_CONFIG["device"]),
                )

                base_row = {
                    "condition": condition,
                    "seed": seed,
                    "method": method,
                    "test_image_id": int(image_id),
                    "predicted_class": int(row["predicted_class"]),
                    "true_class": int(row["true_class"]),
                    "correct": bool(row["correct"]),
                    "insertion_auc": insertion_auc(result.insertion_scores, x=result.x),
                    "deletion_auc": deletion_auc(result.deletion_scores, x=result.x),
                    "drop_top10": float(result.drop_top10),
                    "drop_top20": float(result.drop_top20),
                    "flip_top10": bool(result.flip_at_top10),
                    "flip_top20": bool(result.flip_at_top20),
                }

                per_image_rows.append({**base_row, "slice": "all200"})
                if image_id in primary_ids:
                    per_image_rows.append({**base_row, "slice": "primary"})

per_image_df = pd.DataFrame(per_image_rows)
if per_image_df.empty:
    raise RuntimeError("No per-image rows were generated.")

seed_summary_rows = summarize_seed_level_metrics(per_image_rows)
condition_summary_rows = summarize_condition_level_metrics(seed_summary_rows)
seed_summary_df = pd.DataFrame(seed_summary_rows)
condition_summary_df = pd.DataFrame(condition_summary_rows)
display(seed_summary_df.head())
display(condition_summary_df)

faithfulness_root = ARTIFACTS_ROOT_RESOLVED / "metrics" / "faithfulness"
per_image_csv = faithfulness_root / "per_image_scores.csv"
seed_summary_csv = faithfulness_root / "seed_level_scores.csv"
condition_summary_csv = faithfulness_root / "condition_summary.csv"
per_image_json = faithfulness_root / "per_image_scores.json"

ensure_parent(per_image_csv)
per_image_df.to_csv(per_image_csv, index=False)
seed_summary_df.to_csv(seed_summary_csv, index=False)
condition_summary_df.to_csv(condition_summary_csv, index=False)
write_json(per_image_json, per_image_rows)

paired_rows: list[dict[str, object]] = []
primary_df = per_image_df[per_image_df["slice"] == "primary"]
for method in list(RUN_CONFIG["methods"]):
    method_df = primary_df[primary_df["method"] == method]
    for seed in [int(s) for s in list(RUN_CONFIG["seeds"])]:
        seed_df = method_df[method_df["seed"] == seed]
        for condition_a, condition_b in combinations(list(RUN_CONFIG["conditions"]), 2):
            a_df = seed_df[seed_df["condition"] == condition_a][["test_image_id", "insertion_auc", "deletion_auc"]]
            b_df = seed_df[seed_df["condition"] == condition_b][["test_image_id", "insertion_auc", "deletion_auc"]]
            merged = a_df.merge(b_df, on="test_image_id", suffixes=("_a", "_b"))
            if merged.empty:
                continue

            ins_mean, ins_ci_low, ins_ci_high = paired_bootstrap_ci(
                merged["insertion_auc_a"].to_numpy(),
                merged["insertion_auc_b"].to_numpy(),
                num_resamples=int(RUN_CONFIG["bootstrap_resamples"]),
                seed=seed,
            )
            del_mean, del_ci_low, del_ci_high = paired_bootstrap_ci(
                merged["deletion_auc_a"].to_numpy(),
                merged["deletion_auc_b"].to_numpy(),
                num_resamples=int(RUN_CONFIG["bootstrap_resamples"]),
                seed=seed,
            )

            paired_rows.append(
                {
                    "method": method,
                    "seed": seed,
                    "condition_a": condition_a,
                    "condition_b": condition_b,
                    "n_images": int(merged.shape[0]),
                    "delta_ins_mean": float(ins_mean),
                    "delta_ins_ci_low": float(ins_ci_low),
                    "delta_ins_ci_high": float(ins_ci_high),
                    "delta_ins_pvalue": float(
                        paired_permutation_pvalue(
                            merged["insertion_auc_a"].to_numpy(),
                            merged["insertion_auc_b"].to_numpy(),
                            num_resamples=int(RUN_CONFIG["permutation_resamples"]),
                            seed=seed,
                        )
                    ),
                    "delta_del_mean": float(del_mean),
                    "delta_del_ci_low": float(del_ci_low),
                    "delta_del_ci_high": float(del_ci_high),
                    "delta_del_pvalue": float(
                        paired_permutation_pvalue(
                            merged["deletion_auc_a"].to_numpy(),
                            merged["deletion_auc_b"].to_numpy(),
                            num_resamples=int(RUN_CONFIG["permutation_resamples"]),
                            seed=seed,
                        )
                    ),
                }
            )

paired_df = pd.DataFrame(paired_rows)
paired_csv = faithfulness_root / "paired_stats_primary.csv"
paired_df.to_csv(paired_csv, index=False)
if not paired_df.empty:
    display(paired_df.head())

fig_root = faithfulness_root / "figures"
ensure_parent(fig_root / "placeholder.txt")

primary_summary = condition_summary_df[condition_summary_df["slice"] == "primary"]
methods = list(RUN_CONFIG["methods"])
conditions = list(RUN_CONFIG["conditions"])
x = np.arange(len(conditions))
width = 0.25
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
for idx, method in enumerate(methods):
    offset = (idx - 1) * width
    method_df = primary_summary[primary_summary["method"] == method].set_index("condition")
    ins_vals = [float(method_df.loc[c, "mean_insertion_auc"]) for c in conditions]
    ins_err = [float(method_df.loc[c, "std_insertion_auc"]) for c in conditions]
    del_vals = [float(method_df.loc[c, "mean_deletion_auc"]) for c in conditions]
    del_err = [float(method_df.loc[c, "std_deletion_auc"]) for c in conditions]
    axes[0].bar(x + offset, ins_vals, width=width, yerr=ins_err, capsize=3, label=method)
    axes[1].bar(x + offset, del_vals, width=width, yerr=del_err, capsize=3, label=method)

axes[0].set_title("Primary Slice Insertion AUC (higher better)")
axes[1].set_title("Primary Slice Deletion AUC (lower better)")
for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(conditions)
    ax.grid(axis="y", alpha=0.25)
axes[0].legend()
auc_fig_path = fig_root / "primary_auc_by_condition_method.png"
fig.savefig(auc_fig_path, dpi=200, bbox_inches="tight")
plt.show()

gradcam_seed = seed_summary_df[(seed_summary_df["slice"] == "primary") & (seed_summary_df["method"] == "gradcam")][["condition", "seed", "mean_insertion_auc", "mean_deletion_auc"]]
acc_rows = []
for run in runs:
    payload = read_json(run.run_metrics_path)
    acc_rows.append({"condition": run.condition, "seed": int(run.seed), "test_acc": float(payload["test_acc"])})
acc_df = pd.DataFrame(acc_rows)
merged_df = gradcam_seed.merge(acc_df, on=["condition", "seed"], how="inner")
fig, ax = plt.subplots(figsize=(6, 4))
for condition in conditions:
    sub = merged_df[merged_df["condition"] == condition]
    ax.scatter(sub["test_acc"], sub["mean_insertion_auc"], label=condition)
ax.set_xlabel("Test Accuracy")
ax.set_ylabel("Primary Insertion AUC (Grad-CAM)")
ax.set_title("Accuracy vs Faithfulness")
ax.grid(alpha=0.25)
ax.legend()
acc_fig_path = fig_root / "accuracy_vs_faithfulness.png"
fig.savefig(acc_fig_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved Stage-7 artifacts under: {faithfulness_root}")


## Stage 8 Partial Fine-Tuning Ablation


In [ ]:
ablation_modes = {"ablation_layer4", "ablation_mode", "limited_finetune"}
stage8_conditions = [str(condition) for condition in list(RUN_CONFIG["stage8_conditions"])]
stage8_seed = int(RUN_CONFIG["stage8_seed"])
stage8_recipe_id = str(RUN_CONFIG["stage8_recipe_id"])

stage8_run_results: list[dict[str, object]] = []
if RUN_CONFIG["run_stage8_ablation"]:
    print("Running Stage-8 ablation training runs...")
    for condition in stage8_conditions:
        config = TrainingRunConfig(
            condition=condition,
            seed=stage8_seed,
            recipe_id=stage8_recipe_id,
            device=str(RUN_CONFIG["device"]),
            artifacts_root=RUN_CONFIG["artifacts_root"],
            data_root=RUN_CONFIG["data_root"],
            download=False,
            num_workers=int(RUN_CONFIG["stage8_num_workers"]),
            pin_memory=bool(RUN_CONFIG["stage8_pin_memory"]),
            allow_remote_download=bool(RUN_CONFIG["allow_remote_download"]),
            moco_checkpoint_path=str(DEFAULT_ENCODER_CHECKPOINTS.moco_checkpoint_path),
            swav_checkpoint_path=str(DEFAULT_ENCODER_CHECKPOINTS.swav_checkpoint_path),
            sanity_checks=bool(RUN_CONFIG["stage8_sanity_checks"]),
        )
        print(f"[stage8-run] condition={condition} seed={stage8_seed} recipe={stage8_recipe_id}")
        result = train_one_run(config)
        stage8_run_results.append(result)
        print(
            f"[stage8-done] condition={condition} best_val_acc={result['best_val_acc']:.4f} test_acc={result['test_acc']:.4f}"
        )
else:
    print("Skipping Stage-8 run generation (run_stage8_ablation=False).")

run_metrics_root = ARTIFACTS_ROOT_RESOLVED / "metrics" / "probe_runs"
ablation_rows: list[dict[str, object]] = []
baseline_rows: list[dict[str, object]] = []
if run_metrics_root.exists():
    for path in sorted(run_metrics_root.glob("*/*.json")):
        payload = read_json(path)
        if not isinstance(payload, dict):
            continue

        condition = payload.get("condition")
        if condition not in stage8_conditions:
            continue
        if int(payload.get("seed", -1)) != stage8_seed:
            continue

        row = {
            "condition": str(payload["condition"]),
            "seed": int(payload["seed"]),
            "training_mode": str(payload["training_mode"]),
            "recipe_id": str(payload["recipe_id"]),
            "test_acc": float(payload["test_acc"]),
            "best_val_acc": float(payload["best_val_acc"]),
            "metrics_path": str(path),
        }
        if row["training_mode"] == "frozen_probe":
            baseline_rows.append(row)
        elif row["training_mode"] in ablation_modes:
            ablation_rows.append(row)

if not baseline_rows or not ablation_rows:
    print("Stage-8 comparison incomplete. Need both frozen_probe and ablation runs for selected conditions.")
else:
    baseline_df = (
        pd.DataFrame(baseline_rows)
        .sort_values(["condition", "recipe_id", "metrics_path"])
        .drop_duplicates("condition", keep="last")
    )
    ablation_df = (
        pd.DataFrame(ablation_rows)
        .sort_values(["condition", "recipe_id", "metrics_path"])
        .drop_duplicates("condition", keep="last")
    )

    merged = baseline_df.merge(ablation_df, on="condition", suffixes=("_frozen", "_ablation"))
    merged["delta_test_acc"] = merged["test_acc_ablation"] - merged["test_acc_frozen"]
    merged["delta_best_val_acc"] = merged["best_val_acc_ablation"] - merged["best_val_acc_frozen"]
    display(merged)

    ablation_root = ARTIFACTS_ROOT_RESOLVED / "metrics" / "ablation_layer4"
    comparison_csv = ablation_root / f"comparison_seed{stage8_seed}.csv"
    comparison_json = ablation_root / f"comparison_seed{stage8_seed}.json"
    ensure_parent(comparison_csv)
    merged.to_csv(comparison_csv, index=False)
    write_json(comparison_json, merged.to_dict(orient="records"))

    ranking = merged.sort_values("test_acc_ablation", ascending=False)[["condition", "test_acc_ablation"]]
    write_json(
        ablation_root / f"ablation_ranking_seed{stage8_seed}.json",
        ranking.to_dict(orient="records"),
    )

    fig, ax = plt.subplots(figsize=(6.5, 4))
    x = np.arange(merged.shape[0])
    w = 0.35
    ax.bar(x - w / 2, merged["test_acc_frozen"], width=w, label="frozen_probe")
    ax.bar(x + w / 2, merged["test_acc_ablation"], width=w, label="ablation_layer4")
    ax.set_xticks(x)
    ax.set_xticklabels(merged["condition"].tolist())
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel("Test Accuracy")
    ax.set_title(f"Stage-8 Optional Ablation (Seed {stage8_seed})")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.savefig(ablation_root / "ablation_vs_frozen_acc.png", dpi=200, bbox_inches="tight")
    plt.show()

    print(f"Saved Stage-8 dedicated summary under: {ablation_root}")


## Stage 9 - Mandatory Grad-CAM++ Diagnostics


In [ ]:
primary_rows = [
    row for row in per_image_rows if row["slice"] == "primary" and row["method"] in {"gradcam", "gradcampp"}
]
delta_rows = compute_method_deltas(primary_rows, method_a="gradcam", method_b="gradcampp")
seed_delta_rows = summarize_seed_level_deltas(delta_rows)
condition_delta_rows = summarize_condition_level_deltas(seed_delta_rows)

seed_delta_df = pd.DataFrame(seed_delta_rows)
condition_delta_df = pd.DataFrame(condition_delta_rows)

seed_method_df = seed_summary_df[
    (seed_summary_df["slice"] == "primary") & seed_summary_df["method"].isin(["gradcam", "gradcampp"])
]
gradcam_df = seed_method_df[seed_method_df["method"] == "gradcam"].rename(
    columns={"mean_insertion_auc": "gradcam_ins", "mean_deletion_auc": "gradcam_del"}
)[["condition", "seed", "gradcam_ins", "gradcam_del"]]
gradcampp_df = seed_method_df[seed_method_df["method"] == "gradcampp"].rename(
    columns={"mean_insertion_auc": "gradcampp_ins", "mean_deletion_auc": "gradcampp_del"}
)[["condition", "seed", "gradcampp_ins", "gradcampp_del"]]

n_primary_df = seed_method_df[seed_method_df["method"] == "gradcam"][["condition", "seed", "n_images"]].rename(
    columns={"n_images": "n_primary"}
)

seed_method_delta_df = gradcam_df.merge(gradcampp_df, on=["condition", "seed"], how="inner")
seed_method_delta_df = seed_method_delta_df.merge(n_primary_df, on=["condition", "seed"], how="left")
seed_method_delta_df["delta_ins"] = seed_method_delta_df["gradcampp_ins"] - seed_method_delta_df["gradcam_ins"]
seed_method_delta_df["delta_del"] = seed_method_delta_df["gradcampp_del"] - seed_method_delta_df["gradcam_del"]
seed_method_delta_df = seed_method_delta_df[
    [
        "condition",
        "seed",
        "n_primary",
        "gradcam_ins",
        "gradcampp_ins",
        "delta_ins",
        "gradcam_del",
        "gradcampp_del",
        "delta_del",
    ]
]

display(seed_method_delta_df)
display(seed_delta_df)
display(condition_delta_df)

primary_condition_deltas = [row for row in condition_delta_rows if row["slice"] == "primary"]
outcome_label = classify_gradcampp_outcome(
    primary_condition_deltas,
    threshold=float(RUN_CONFIG["gradcampp_effect_threshold"]),
)

label_to_statement = {
    "reinforce": "Grad-CAM++ diagnostics reinforce the primary interpretation.",
    "neutral": "Grad-CAM++ diagnostics are mostly neutral to the primary interpretation.",
    "weaken": "Grad-CAM++ diagnostics weaken the primary interpretation.",
    "mixed": "Grad-CAM++ diagnostics are mixed across conditions.",
}
diagnostics_note = label_to_statement[outcome_label]
print(f"Stage-9 Grad-CAM++ outcome label: {outcome_label}")
print(diagnostics_note)

diag_root = ARTIFACTS_ROOT_RESOLVED / "metrics" / "gradcampp_diagnostics"
seed_method_delta_csv = diag_root / "seed_level_method_and_delta_scores.csv"
seed_delta_csv = diag_root / "seed_level_deltas.csv"
condition_delta_csv = diag_root / "condition_level_deltas.csv"
outcome_label_json = diag_root / "outcome_label.json"
diagnostics_note_json = diag_root / "diagnostics_note.json"

ensure_parent(seed_method_delta_csv)
seed_method_delta_df.to_csv(seed_method_delta_csv, index=False)
seed_delta_df.to_csv(seed_delta_csv, index=False)
condition_delta_df.to_csv(condition_delta_csv, index=False)
write_json(
    outcome_label_json,
    {
        "label": outcome_label,
        "threshold": float(RUN_CONFIG["gradcampp_effect_threshold"]),
        "slice": "primary",
    },
)
write_json(
    diagnostics_note_json,
    {
        "label": outcome_label,
        "note": diagnostics_note,
        "threshold": float(RUN_CONFIG["gradcampp_effect_threshold"]),
        "n_primary_by_condition_seed": seed_method_delta_df[["condition", "seed", "n_primary"]].to_dict(orient="records"),
    },
)

print("Saved Stage-9 quantitative outputs:")
print(f"- seed method+delta CSV: {seed_method_delta_csv}")
print(f"- seed delta CSV:        {seed_delta_csv}")
print(f"- condition delta CSV:   {condition_delta_csv}")
print(f"- outcome label JSON:    {outcome_label_json}")
print(f"- diagnostics note JSON: {diagnostics_note_json}")


In [ ]:
def _denormalize_image(image_tensor: torch.Tensor) -> np.ndarray:
    mean = np.asarray([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.asarray([0.229, 0.224, 0.225], dtype=np.float32)
    image = image_tensor.detach().cpu().permute(1, 2, 0).numpy()
    image = image * std + mean
    return np.clip(image, 0.0, 1.0)

panel_seed = int(RUN_CONFIG["stage9_panel_seed"])
panel_ids = RUN_CONFIG["stage9_panel_image_ids"]
if panel_ids is None:
    panel_reference_condition = str(RUN_CONFIG["stage9_panel_reference_condition"])
    panel_size = int(RUN_CONFIG["stage9_panel_num_images"])
    ref_rows = metadata_by_key[(panel_reference_condition, panel_seed, "gradcam")]
    ref_indexed = _index_rows_by_image_id(ref_rows)

    correct_ids = [image_id for image_id in subset_ids if bool(ref_indexed[image_id]["correct"]) ]
    incorrect_ids = [image_id for image_id in subset_ids if not bool(ref_indexed[image_id]["correct"]) ]

    selected_ids: list[int] = []
    if correct_ids:
        selected_ids.append(int(correct_ids[0]))
    if incorrect_ids:
        selected_ids.append(int(incorrect_ids[0]))

    for image_id in subset_ids:
        if image_id in selected_ids:
            continue
        selected_ids.append(int(image_id))
        if len(selected_ids) >= panel_size:
            break

    panel_ids = selected_ids[:panel_size]
    print(
        f"Auto-selected panel ids (seed={panel_seed}, reference={panel_reference_condition}): {panel_ids}; includes correct={bool(correct_ids)} incorrect={bool(incorrect_ids)}"
    )
else:
    panel_ids = [int(image_id) for image_id in panel_ids]

if not panel_ids:
    print("No panel image ids available; skipping Stage-9 qualitative panel.")
else:
    panel_methods = ["gradcam", "gradcampp"]
    num_rows = len(list(RUN_CONFIG["conditions"]))
    num_cols = len(panel_ids) * len(panel_methods)
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(4.0 * num_cols, 3.2 * num_rows))
    if num_rows == 1 and num_cols == 1:
        axes = np.asarray([[axes]])
    elif num_rows == 1:
        axes = np.expand_dims(axes, axis=0)
    elif num_cols == 1:
        axes = np.expand_dims(axes, axis=1)

    for row_idx, condition in enumerate(list(RUN_CONFIG["conditions"])):
        for image_offset, image_id in enumerate(panel_ids):
            image_tensor, _ = test_dataset[int(image_id)]
            image_vis = _denormalize_image(image_tensor)

            for method_offset, method in enumerate(panel_methods):
                col_idx = image_offset * len(panel_methods) + method_offset
                ax = axes[row_idx, col_idx]
                rows = metadata_by_key[(condition, panel_seed, method)]
                indexed = _index_rows_by_image_id(rows)
                meta = indexed[int(image_id)]
                saliency = load_saliency_map(_resolve_map_path(str(meta["saliency_map_path"]), ARTIFACTS_ROOT_RESOLVED))

                ax.imshow(image_vis)
                ax.imshow(saliency, cmap="jet", alpha=0.45, vmin=0.0, vmax=1.0)
                ax.set_axis_off()
                ax.set_title(
                    f"{condition} | id={image_id} | {method}\n"
                    f"pred={int(meta['predicted_class'])} true={int(meta['true_class'])} correct={bool(meta['correct'])}",
                    fontsize=9,
                )

    fig.suptitle(f"Stage-9 Main Panel: Grad-CAM vs Grad-CAM++ (seed {panel_seed} shared ids)", fontsize=13)
    panel_path = ARTIFACTS_ROOT_RESOLVED / "saliency" / "gradcampp_diagnostics" / "main_panel" / f"seed{panel_seed}_fixed_ids_panel.png"
    ensure_parent(panel_path)
    fig.savefig(panel_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved Stage-9 main panel to: {panel_path}")


## Optional Appendix Plan (Not Implemented Yet)

If we later add robustness appendixes for Grad-CAM++, we will produce:

- `artifacts/metrics/gradcampp_diagnostics/per_class_deltas.csv`
  - grouped by `condition | seed | class_id`
  - reports per-class `delta_ins`, `delta_del`, and sample count `n`
  - suppresses low-count rows (`n < 10`)
- `artifacts/metrics/gradcampp_diagnostics/error_slice_deltas.csv`
  - grouped by `condition | seed | slice` where slice is `correct` or `incorrect`
  - reports slice-wise deltas and `n`
  - suppresses low-count rows (`n < 10`)

These are supplementary robustness diagnostics only; the core Stage-9 comparison remains the required method-level tables and main qualitative panel.
